# Phase G1: Ginza single-UE CFR, beam sweep, and normal/sleep comparison

## 1. Scope and physical assumptions

This notebook reproduces a single-UE point-to-point link from the authoritative
`cfr_mamimo.ipynb` reference, but using the project's own `ArrayConfig` and
codebook infrastructure.

Key physical choices:
- TX: 128-port cross-polarized UPA built from `ArrayConfig(4, 8, 2, 2)`.
  - project pol0 → Sionna cross component 0 (-45°).
  - project pol1 → Sionna cross component 1 (+45°).
- RX: single dual-polarized (VH) isotropic element.
- Scene: Ginza_012 XML at 3.5 GHz, 100 MHz bandwidth, 290 K.
- Material override: `itu_concrete` objects are replaced with ITU concrete
  (thickness 0.5 m, scattering 0.3, XPD 0.3), matching `cfr_mamimo.ipynb`.
- UE: first valid position from the existing `rx_1000.pkl`, falling back to a
  reproducible seed-36 sample only if necessary.
- PathSolver: one formal run with `synthetic_array=True`, seed 36, max_depth 15,
  LOS + specular + diffuse reflection enabled, refraction/diffraction disabled.
- CFR: center frequency only (`frequencies=[0.0]`), no normalization.
- Beam selection: valid PMI mask, DFT codebook (8 vertical × 32 horizontal
  oversampled spatial beams, i2=4; physical array columns remain 8), best
  beam by effective-channel power `sum(|H @ w_sionna|²)`.
- Normal/sleep: same `H`, same selected PMI, sleep applies the right-half
  physical-port muting mask without renormalization.

Absolute SNR/SINR is **not** computed here; link-budget details are deferred to
Phase G1.1.

In [ ]:
## 2. 导入依赖并初始化运行环境

import os
import pickle
import time
import math

# 在导入 GPU 计算库之前指定第 0 块 GPU；setdefault 不会覆盖外部已有设置。
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import numpy as np
import torch
import mitsuba as mi

# 选择支持 CUDA、自动微分和极化场计算的 Mitsuba 运行后端。
mi.set_variant('cuda_ad_mono_polarized')

import sionna.rt as rt

# 导入项目自己的阵列配置、DFT 码本、静默掩码和 PMI 工具。
from mMIMO_sleep.array_config import ArrayConfig
from mMIMO_sleep.simulation.sionna_array import (
    array_config_to_planar_array,
    weights_to_sionna_precoding,
)
from mMIMO_sleep.codebook.dft import generate_dft_codebook
from mMIMO_sleep.codebook.muting import create_right_half_mask, apply_muting_mask
from mMIMO_sleep.codebook.pmi_mask import (
    create_total_loss_pmi_mask,
    beam_indices_from_mask,
)
from mMIMO_sleep.codebook.pmi import PMI, beam_index_to_pmi

print('Environment ready.')
print(f'Mitsuba variant: {mi.variant()}')
print(f'Torch CUDA available: {torch.cuda.is_available()}')


## 3. `cfr_mamimo.ipynb` parameter audit

From the authoritative `cfr_mamimo.ipynb` (TAP_TCB_Resource/Ginza_012) and the
installed `Lib_xhd.tools.sionna_xhd` source, the exact positional parameter
mappings are:

**TX orientation**
```python
mi.Point3f(0, 15/360 * dr.pi, 0)
```
Because `dr.pi` is the mathematical π, this evaluates to
`15/360 * π = 0.1308996938995747 rad = 7.5°`, **not** 15°.

**`SceneParams("clin", True, True, True, 3.5e9, None, None, clin)`**
| Position | Field | Value in notebook |
|---|---|---|
| 1 | `key` | `"clin"` |
| 2 | `info` | `True` |
| 3 | `show` | `True` |
| 4 | `axes` | `True` |
| 5 | `f` | `3.5e9` Hz |
| 6 | `s` | `None` |
| 7 | `cwin` | `None` |
| 8 | `clin` | `"/home/xhd/.../ginza_1/ginza_1.xml"` → mapped locally |

**`PropagParams(15, True, True, True, False, False, False, False, 36)`**
| Position | Field | Value |
|---|---|---|
| 1 | `max_depth` | 15 |
| 2 | `los` | True |
| 3 | `specular_reflection` | True |
| 4 | `diffuse_reflection` | True |
| 5 | `refraction` | False |
| 6 | `diffraction` | False |
| 7 | `edge_diffraction` | False |
| 8 | `diffraction_lit_region` | False |
| 9 | `seed` | 36 |

**`PathsParams(int(6e6), int(2e6), True)`**
| Position | Field | Value |
|---|---|---|
| 1 | `max_num_paths_per_src` | 6,000,000 |
| 2 | `samples_per_src` | 2,000,000 |
| 3 | `synthetic_array` | True |

**`OFDMParams(288, 100e6, 30e3, 4096, 14, 273, False, False, False, 'tf')`**
| Position | Field | Value |
|---|---|---|
| 1 | `CP_length` | 288 |
| 2 | `bandwidth` | 100 MHz |
| 3 | `subcarrier_spacing` | 30 kHz |
| 4 | `num_subcarriers` | 4096 |
| 5 | `num_time_steps` | 14 |
| 6 | `num_resource_blocks` | 273 |
| 7 | `normalize_delays` | False |
| 8 | `normalize` | False |
| 9 | `reverse_direction` | False |
| 10 | `out_type` | `'tf'` |

For this Phase-G1 notebook we only need the carrier frequency and bandwidth;
the OFDM parameters are recorded for Phase G1.1.

In [ ]:
## 4. Ginza 场景与链路参数

# 参数沿用 cfr_mamimo.ipynb；仅将场景路径映射到当前 RunPod 环境。
scene_xml_path = (
    '/workspace/Study_Sionna/Projects/TAP_TCB_Resource/Ginza_012/'
    'ginza_1/ginza_1.xml'
)
carrier_frequency_hz = 3.5e9
scene_bandwidth_hz = 100e6
temperature_k = 290.0

tx_position = (-122.0, -108.5, 41.0)
# 原 notebook 写法为 mi.Point3f(0, 15/360 * dr.pi, 0)。
# 注意该表达式的实际结果是 0.1308997 rad，也就是 7.5°，并非 15°。
tx_orientation = (0.0, (15.0 / 360.0) * math.pi, 0.0)

# 本阶段只记录发射功率和噪声系数，尚不用于绝对 SNR 计算。
tx_power_dbm = 51.13
noise_figure_db = 5.0
seed = 36
max_depth = 15

# 单 UE 正式仿真的 PathSolver 采样预算。参考 notebook 面向 3000 个 UE，
# 每个源使用 2e6 个样本；这里降为 1e6，以减少时间和显存开销。
samples_per_src = 1_000_000
max_num_paths_per_src = 1_000_000

print('Ginza configuration:')
print(f'  scene_xml_path = {scene_xml_path}')
print(f'  carrier_frequency_hz = {carrier_frequency_hz}')
print(f'  scene_bandwidth_hz = {scene_bandwidth_hz}')
print(f'  temperature_k = {temperature_k}')
print(f'  tx_position = {tx_position}')
print(f'  tx_orientation (rad) = {tx_orientation}')
print(f'  tx_orientation y-deg = {math.degrees(tx_orientation[1])}')
print(f'  tx_power_dBm = {tx_power_dbm}')
print(f'  noise_figure_dB = {noise_figure_db}')
print(f'  seed = {seed}')
print(f'  max_depth = {max_depth}')
print(f'  samples_per_src = {samples_per_src}')
print(f'  max_num_paths_per_src = {max_num_paths_per_src}')


In [ ]:
## 5. 加载场景并覆盖无线材料

print('Loading Ginza scene...')
scene = rt.load_scene(scene_xml_path)

# XML 中的默认参数不一定等于本实验设定，因此显式写入频率、带宽和温度。
scene.frequency = mi.Float(carrier_frequency_hz)
scene.bandwidth = mi.Float(scene_bandwidth_hz)
scene.temperature = mi.Float(temperature_k)

# 复现 cfr_mamimo.ipynb Cell 2 的混凝土材料设置。
# scattering_coefficient 控制散射强度，xpd_coefficient 描述交叉极化去耦。
my_concrete = rt.ITURadioMaterial(
    name='my_concrete',
    itu_type='concrete',
    thickness=mi.Float(0.5),
    scattering_coefficient=0.3,
    xpd_coefficient=0.3,
)

# 找到使用默认 concrete 材料的场景对象，并替换为上面的 my_concrete。
replaced_objects = []
for obj_name, obj in scene.objects.items():
    mat_name = obj.radio_material.name
    if mat_name in ('itu_concrete', 'concrete'):
        obj.radio_material = my_concrete
        replaced_objects.append((obj_name, mat_name))

if not replaced_objects:
    raise RuntimeError(
        'No scene object had material "itu_concrete" or "concrete"; '
        'the material override from cfr_mamimo.ipynb cannot be applied.'
    )

print('Applied my_concrete override to the following objects:')
for obj_name, old_mat in replaced_objects:
    print(f'  {obj_name}: {old_mat} -> my_concrete')

print(f'Total objects in scene: {len(scene.objects)}')
print(f'Number of replaced concrete objects: {len(replaced_objects)}')


In [ ]:
## 6. 配置 TX/RX 阵列

# 逻辑阵列为 4×8 子阵；每个逻辑行展开为 2 个物理行。
# 因而共有 8×8=64 个物理位置，双极化后得到 128 个 TX 端口。
config = ArrayConfig(
    num_subarray_rows=4,
    num_horizontal=8,
    elements_per_subarray=2,
    num_polarizations=2,
)

# 使用 3GPP TR 38.901 阵元方向图和 ±45° cross 极化。
tx_array = array_config_to_planar_array(
    config,
    pattern='tr38901',
    polarization='cross',
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
)

assert tx_array.array_size == 64
assert tx_array.num_ant == 128

# UE 使用单个物理位置、V/H 双极化，因此 RX 端口数为 2。
rx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern='iso',
    polarization='VH',
)

assert rx_array.num_ant == 2

# 将阵列模型和发射机安装到 Sionna RT 场景中。
scene.tx_array = tx_array
scene.rx_array = rx_array

scene.add(rt.Transmitter(name='tx', position=tx_position, orientation=tx_orientation))

print('TX/RX arrays configured:')
print(f'  TX array_size = {tx_array.array_size}')
print(f'  TX num_ant = {tx_array.num_ant}')
print(f'  RX num_ant = {rx_array.num_ant}')
print(f'  project pol0 -> Sionna cross component 0 (-45 deg)')
print(f'  project pol1 -> Sionna cross component 1 (+45 deg)')


## 6b. Physical array parameters vs. codebook beam parameters

The physical TX array topology (`ArrayConfig`) and the DFT codebook's angular
oversampling are **independent, separately configured quantities**. They must
not be conflated:

- `config.num_horizontal = 8` is the number of **physical horizontal element
  columns** in the TX subarray grid (fixed by the antenna hardware / ArrayConfig).
- `NUM_HORIZONTAL_BEAMS = 32` is the number of **horizontal DFT codebook
  beams** (i.e. the number of distinct `i11` values). This is a codebook
  design choice, independent of the physical column count.
- horizontal oversampling factor = `NUM_HORIZONTAL_BEAMS / config.num_horizontal`
  = `32 / 8 = 4`.
- `NUM_VERTICAL_BEAMS = 8` (i12 values) and `NUM_I2 = 4` (co-phasing values)
  are unchanged from the physical vertical subarray count and the standard
  4-point co-phasing grid, respectively.

`config.num_horizontal` must **not** be used to set `NUM_HORIZONTAL_BEAMS`;
they serve different roles in `generate_dft_codebook`
(`config.num_horizontal` sets the physical steering-vector length,
`num_horizontal_beams` sets the number of oversampled DFT directions sampled
over that physical aperture).

In [ ]:
## 6c. 码本波束数（与物理阵列尺寸相互独立）

# 垂直采样 8 个波束，水平采样 32 个波束，并使用 4 个 i2 共相位值。
# 水平物理列数只有 8，因此水平方向的过采样倍数为 32/8=4。
NUM_VERTICAL_BEAMS = 8
NUM_HORIZONTAL_BEAMS = 32
NUM_I2 = 4

horizontal_oversampling_factor = NUM_HORIZONTAL_BEAMS / config.num_horizontal

print('Physical array topology (ArrayConfig):')
print(f'  config.num_subarray_rows = {config.num_subarray_rows}')
print(f'  config.num_horizontal (physical columns) = {config.num_horizontal}')
print(f'  config.elements_per_subarray = {config.elements_per_subarray}')
print(f'  config.num_polarizations = {config.num_polarizations}')
print()
print('Codebook beam-count parameters (independent of physical topology):')
print(f'  NUM_VERTICAL_BEAMS = {NUM_VERTICAL_BEAMS}')
print(f'  NUM_HORIZONTAL_BEAMS = {NUM_HORIZONTAL_BEAMS}')
print(f'  NUM_I2 = {NUM_I2}')
print(f'  horizontal oversampling factor = {NUM_HORIZONTAL_BEAMS}/{config.num_horizontal} = {horizontal_oversampling_factor}')


In [ ]:
## 7. 选择并放置 UE

# 优先复用 cfr_mamimo 已生成的 UE 位置集合，以便与参考实验对齐。
pkl_path = (
    '/workspace/Study_Sionna/Projects/TAP_TCB_Resource/Ginza_012/'
    'Results/rx_1000.pkl'
)

with open(pkl_path, 'rb') as f:
    ue_data = pickle.load(f)

loaded_positions = [np.array(p, dtype=np.float64) for p in ue_data['all_rx_positions']]
print(f'Loaded {len(loaded_positions)} UE positions from {pkl_path}')


# 使用较小采样数快速探测候选位置是否至少存在一条有效传播路径。
# 函数名保留历史写法；返回 True 只表示存在有效路径，不保证一定存在 LOS。
def has_valid_los_path(scene, position, solver=None):
    '''Lightweight probe: return True if at least one path is found.'''
    if solver is None:
        solver = rt.PathSolver()
    # 临时加入探测用 RX；无论求解是否成功，finally 都会将其移除。
    scene.add(rt.Receiver(name='rx_probe', position=position.tolist()))
    try:
        paths = solver(
            scene,
            max_depth=max_depth,
            samples_per_src=1000,
            synthetic_array=True,
            los=True,
            specular_reflection=True,
            diffuse_reflection=True,
            refraction=False,
            diffraction=False,
            edge_diffraction=False,
            diffraction_lit_region=False,
            seed=seed,
        )
        return bool(np.asarray(paths.valid).any())
    finally:
        scene.remove('rx_probe')


ue_position = None
ue_source = None
ue_index = -1

# 按文件中的既有顺序查找第一个可用 UE，并复用同一个轻量探测器。
probe_solver = rt.PathSolver()
for idx, pos in enumerate(loaded_positions):
    if has_valid_los_path(scene, pos, solver=probe_solver):
        ue_position = pos
        ue_source = f'rx_1000.pkl index {idx}'
        ue_index = idx
        break

if ue_position is None:
    # 如果预生成位置均不可用，则在参考 notebook 的 700×700 m 区域内，
    # 使用固定随机种子采样，直到找到存在有效路径的位置。
    rng = np.random.default_rng(seed)
    half_range = 350.0
    for attempt in range(5000):
        x = tx_position[0] + rng.uniform(-half_range, half_range)
        y = tx_position[1] + rng.uniform(-half_range, half_range)
        pos = np.array([x, y, 1.5], dtype=np.float64)
        if has_valid_los_path(scene, pos, solver=probe_solver):
            ue_position = pos
            ue_source = f'seed-{seed} random sample attempt {attempt}'
            break
    if ue_position is None:
        raise RuntimeError('Could not find any valid UE position.')

# 将最终选择的 UE 加入正式场景，并计算其与 TX 的二维/三维距离。
scene.add(rt.Receiver(name='rx', position=ue_position.tolist()))

tx_pos_arr = np.array(tx_position, dtype=np.float64)
horizontal_dist = float(np.linalg.norm(ue_position[:2] - tx_pos_arr[:2]))
dist_3d = float(np.linalg.norm(ue_position - tx_pos_arr))

print('UE placement:')
print(f'  position = {ue_position}')
print(f'  source = {ue_source}')
print(f'  TX-UE horizontal distance = {horizontal_dist:.3f} m')
print(f'  TX-UE 3D distance = {dist_3d:.3f} m')


In [ ]:
## 8. 正式运行 PathSolver

# synthetic_array=True：仅对阵列参考点追踪几何路径，再为所有端口
# 合成相位响应，可显著降低 128 端口阵列的追踪成本。
path_solver_kwargs = {
    'max_depth': max_depth,
    'max_num_paths_per_src': max_num_paths_per_src,
    'samples_per_src': samples_per_src,
    'synthetic_array': True,
    'los': True,
    'specular_reflection': True,
    'diffuse_reflection': True,
    'refraction': False,
    'diffraction': False,
    'edge_diffraction': False,
    'diffraction_lit_region': False,
    'seed': seed,
}

print('Formal PathSolver arguments:')
for k, v in path_solver_kwargs.items():
    print(f'  {k} = {v}')

# 计时说明：rt.PathSolver()(...) 可能只把任务派发给 Dr.Jit/CUDA，
# 返回时结果尚未真正写回主机。因此在同一个 paths 对象上分三段计时，
# 不重复运行求解器：
#   1. dispatch：PathSolver 调用返回所需时间；
#   2. materialization：读取 valid/tau/a，强制 GPU 完成计算并返回结果；
#   3. CFR materialization：下一单元中生成 CFR 的累计时间。
rt_t0 = time.time()
paths = rt.PathSolver()(scene, **path_solver_kwargs)
rt_t_dispatch = time.time()
dispatch_time = rt_t_dispatch - rt_t0

# 将惰性 Dr.Jit 数组转换为 NumPy，以强制完成计算并取得有效路径、时延和增益。
valid_mask = np.asarray(paths.valid)
valid_count = int(valid_mask.sum())
tau = np.asarray(paths.tau)
tau_min = float(tau.min()) if valid_count else float('nan')
tau_max = float(tau.max()) if valid_count else float('nan')
a0 = np.asarray(paths.a[0])
rt_t_materialized = time.time()
materialize_time = rt_t_materialized - rt_t_dispatch
total_time_to_materialized = rt_t_materialized - rt_t0

print(f'PathSolver call/dispatch time: {dispatch_time:.3f} s')
print(f'PathSolver materialization time (valid/tau/a): {materialize_time:.3f} s')
print(f'Total time through path materialization: {total_time_to_materialized:.3f} s')
print(f'valid path count: {valid_count}')
print(f'tau shape: {tau.shape}')
print(f'tau min/max: {tau_min:.3e} / {tau_max:.3e} s')
print(f'paths.a[0] shape: {a0.shape}')

if valid_count == 0:
    raise RuntimeError('No valid paths found; stopping.')


In [ ]:
## 9. 提取 CFR 和信道矩阵 H

# 在相对载频 0 Hz（中心频点）计算 CFR；不归一化增益，也不归一化时延。
cfr_t0 = time.time()
h_freq = paths.cfr(
    frequencies=mi.Float([0.0]),
    normalize=False,
    normalize_delays=False,
    out_type='numpy',
)
# 原始 CFR 含场景批次、TX/RX 数量和频点等单例维度。
# squeeze 后保留 [RX port, TX port]，本实验应为 [2, 128]。
H = np.asarray(h_freq).squeeze()
cfr_t1 = time.time()
cfr_time = cfr_t1 - cfr_t0
total_time_through_cfr = cfr_t1 - rt_t0

print(f'Raw CFR shape: {np.asarray(h_freq).shape}')
print(f'H shape after squeeze: {H.shape}')
print(f'CFR materialization time (paths.cfr + squeeze): {cfr_time:.3f} s')
print(f'Total time through CFR materialization (from PathSolver call): {total_time_through_cfr:.3f} s')

assert H.shape == (2, 128), f'Expected H.shape == (2, 128), got {H.shape}'
assert np.isfinite(H).all(), 'H contains non-finite values'

# 转为 PyTorch 复数张量，但不改变 Sionna 的 TX/RX 端口顺序。
H_t = torch.from_numpy(H).to(torch.complex64)

print('Final H:')
print(f'  dtype = {H_t.dtype}')
print(f'  shape = {H_t.shape}')
print(f'  finite = {torch.isfinite(H_t).all().item()}')


In [ ]:
## 10. 在有效码本波束中执行波束扫描

# 这里使用独立定义的码本波束数 8×32×4，而不是物理水平列数
# config.num_horizontal=8。最终码本包含 1024 个、每个长度 128 的码字。
codebook = generate_dft_codebook(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    device='cpu',
)
print(f'Codebook shape: {codebook.shape}')
assert codebook.shape == (1024, 128), f'Expected (1024, 128), got {codebook.shape}'

# 有效 PMI mask 的轴顺序是 [i12, i11]；它筛选 normal/sleep
# 峰值损失满足目标范围的空间 PMI。i2 不在该二维 mask 中。
valid_pmi_mask = create_total_loss_pmi_mask(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    device='cpu',
)
print(f'Valid PMI mask shape: {valid_pmi_mask.shape}')
assert valid_pmi_mask.shape == (8, 32), f'Expected (8, 32), got {valid_pmi_mask.shape}'

# 将每个有效空间 PMI 展开为 NUM_I2 个完整波束索引。
valid_spatial_pmi_count = int(valid_pmi_mask.sum())
valid_beam_indices = beam_indices_from_mask(valid_pmi_mask, num_i2=NUM_I2)
valid_full_beam_count = len(valid_beam_indices)

assert valid_full_beam_count == valid_spatial_pmi_count * NUM_I2, (
    f'valid_full_beam_count ({valid_full_beam_count}) != '
    f'valid_spatial_pmi_count * NUM_I2 '
    f'({valid_spatial_pmi_count} * {NUM_I2} = {valid_spatial_pmi_count * NUM_I2})'
)

print(f'Valid spatial PMI count: {valid_spatial_pmi_count}')
print(f'Valid full beam count: {valid_full_beam_count}')

# 只扫描有效的完整波束。项目码字采用 polarization-major + row-major，
# 而 Sionna 阵元采用 column-first，因此先执行端口重排，再与 H 相乘。
# 对每个波束计算 g=H@w，并以两个 RX 极化端口的总功率 sum(|g|²) 作为评分。
best_score = -float('inf')
best_flat_idx = -1

for flat_idx in valid_beam_indices:
    w_project = codebook[flat_idx]
    real, imag = weights_to_sionna_precoding(
        w_project.unsqueeze(0), config
    )
    w_sionna = torch.complex(real, imag).squeeze(0)
    g = H_t @ w_sionna
    score = torch.sum(torch.abs(g) ** 2).item()
    if score > best_score:
        best_score = score
        best_flat_idx = flat_idx

# 将最优的一维 flat beam index 还原为 PMI(i11, i12, i2)。
selected_pmi = beam_index_to_pmi(
    best_flat_idx,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_i2=NUM_I2,
)

assert 0 <= selected_pmi.i11 < NUM_HORIZONTAL_BEAMS
assert 0 <= selected_pmi.i12 < NUM_VERTICAL_BEAMS
assert 0 <= selected_pmi.i2 < NUM_I2

# 独立验证展平规则：i2 变化最快，其次 i11，最后 i12。
expected_index = (
    selected_pmi.i12 * NUM_HORIZONTAL_BEAMS + selected_pmi.i11
) * NUM_I2 + selected_pmi.i2
assert best_flat_idx == expected_index, (
    f'best_flat_idx ({best_flat_idx}) != expected_index ({expected_index})'
)

print('Beam sweep result:')
print(f'  valid spatial PMI count = {valid_spatial_pmi_count}')
print(f'  valid full beam count = {valid_full_beam_count}')
print(f'  selected flat beam index = {best_flat_idx}')
print(f'  selected PMI = {selected_pmi}')
print(f'  expected_index (formula check) = {expected_index}')
print(f'  selected score = {best_score}')


In [ ]:
## 11. 在同一个 H 上比较 normal 和 sleep

# normal 使用最优完整码字；sleep 将阵列右半边端口置零，且不重新归一化。
w_normal_project = codebook[best_flat_idx]
right_half_mask = create_right_half_mask(config, device='cpu')
w_sleep_project = apply_muting_mask(w_normal_project, right_half_mask)

# 码字模平方之和代表相对发射功率：normal≈1，sleep≈0.5。
norm_normal = torch.sum(torch.abs(w_normal_project) ** 2).item()
norm_sleep = torch.sum(torch.abs(w_sleep_project) ** 2).item()

print('Weight norms:')
print(f'  ||w_normal||^2 = {norm_normal:.6f}')
print(f'  ||w_sleep||^2 = {norm_sleep:.6f}')

# 将项目端口顺序转换为 Sionna 顺序，并由实部/虚部重新组成复数权重。
real_n, imag_n = weights_to_sionna_precoding(
    w_normal_project.unsqueeze(0), config
)
w_normal_sionna = torch.complex(real_n, imag_n).squeeze(0)

real_s, imag_s = weights_to_sionna_precoding(
    w_sleep_project.unsqueeze(0), config
)
w_sleep_sionna = torch.complex(real_s, imag_s).squeeze(0)

# 使用完全相同的信道矩阵 H，分别计算两个 RX 极化端口的有效信道。
g_normal = H_t @ w_normal_sionna
g_sleep = H_t @ w_sleep_sionna

# 对两个 RX 端口的功率求和；该量与理想 MRC 后的接收 SNR 成正比。
power_normal = torch.sum(torch.abs(g_normal) ** 2).item()
power_sleep = torch.sum(torch.abs(g_sleep) ** 2).item()
# 功率比转换为 dB。正值表示 sleep 模式的有效信道功率更低。
loss_db = 10.0 * math.log10(power_normal / power_sleep)

print('Effective-channel powers (same H):')
print(f'  power_normal = {power_normal}')
print(f'  power_sleep = {power_sleep}')
print(f'  same-H loss = {loss_db:.4f} dB')


In [ ]:
## 12. 汇总断言：验证整条数据流的一致性

# 检查信道维度、数值有限性、有效波束以及 normal/sleep 码字功率。
assert H_t.shape == (2, 128)
assert torch.isfinite(H_t).all()
assert valid_spatial_pmi_count > 0
assert 0 <= best_flat_idx < codebook.shape[0]
assert abs(norm_normal - 1.0) < 1e-4
assert abs(norm_sleep - 0.5) < 1e-4

# PMI 范围由码本波束数决定，而不是由物理阵列的行列数决定。
assert 0 <= selected_pmi.i11 < NUM_HORIZONTAL_BEAMS
assert 0 <= selected_pmi.i12 < NUM_VERTICAL_BEAMS
assert 0 <= selected_pmi.i2 < NUM_I2

# 再次独立检查 flat index 与 (i11, i12, i2) 之间的映射公式。
expected_index_check = (
    selected_pmi.i12 * NUM_HORIZONTAL_BEAMS + selected_pmi.i11
) * NUM_I2 + selected_pmi.i2
assert best_flat_idx == expected_index_check

# 重新计算最优 normal 功率，并确认它与波束扫描记录的 best_score 一致。
score_normal = torch.sum(torch.abs(H_t @ w_normal_sionna) ** 2).item()
assert abs(score_normal - best_score) < 1e-5 * max(best_score, 1.0)

# normal 与 sleep 必须使用同一个 PMI；二者只允许在静默掩码上不同。
assert torch.equal(w_normal_project, codebook[best_flat_idx])
assert torch.equal(w_sleep_project, apply_muting_mask(w_normal_project, right_half_mask))

# 最终功率必须为有限正数，损失值也必须是有限数。
assert math.isfinite(power_normal) and power_normal > 0
assert math.isfinite(power_sleep) and power_sleep > 0
assert math.isfinite(loss_db)

print('ALL SUMMARY ASSERTIONS PASSED')
print(f'  NUM_VERTICAL_BEAMS={NUM_VERTICAL_BEAMS}, NUM_HORIZONTAL_BEAMS={NUM_HORIZONTAL_BEAMS}, NUM_I2={NUM_I2}')
print(f'  Codebook shape: {tuple(codebook.shape)}')
print(f'  Valid PMI mask shape: {tuple(valid_pmi_mask.shape)}')
print(f'  Valid spatial PMI count: {valid_spatial_pmi_count}')
print(f'  Valid full beam count: {valid_full_beam_count}')
print(f'  Selected flat beam index: {best_flat_idx}')
print(f'  Selected PMI: i11={selected_pmi.i11}, i12={selected_pmi.i12}, i2={selected_pmi.i2}')
print(f'  Normal power: {power_normal}')
print(f'  Sleep power: {power_sleep}')
print(f'  Same-H loss: {loss_db:.4f} dB')


## 12b. Why the Phase-G1-fix result matches the superseded 8×8 result

The Phase-G1-fix beam sweep (`NUM_HORIZONTAL_BEAMS = 32`) selected
`PMI(i11=24, i12=1, i2=0)`, while the superseded Phase-G1 sweep (which
incorrectly used `NUM_HORIZONTAL_BEAMS = 8`, conflating it with the physical
column count `config.num_horizontal`) selected `PMI(i11=6, i12=1, i2=0)`.

The near-identical `power_normal`, `power_sleep`, and `same-H loss` values
between these two runs are **not a numerical coincidence** and **not caused
by reusing the old result**. They arise because both PMIs correspond to the
same normalized horizontal DFT spatial frequency, i.e. the same physical
steering direction:

```
i11_old / Kh_old = 6 / 8  = 0.75
i11_new / Kh_new = 24 / 32 = 0.75
```

A horizontal DFT beam's steering direction is determined only by the ratio
`i11 / num_horizontal_beams` (the fractional cycles per physical element in
`dft_vector`), not by `i11` or `num_horizontal_beams` individually. Since
`6/8` and `24/32` are the same fraction, the two PMIs steer the physical
8-column aperture toward the same direction, producing the same array
response and therefore the same effective-channel power and same-H loss.
This is expected: the 32-beam codebook is a 4x oversampled refinement of the
8-beam grid, and `i11=24` is exactly the oversampled-grid beam that coincides
with the old `i11=6` on the coarser grid.

## 13. Deferred link-budget items (Phase G1.1)

The following quantities are recorded but **not** used for absolute SNR/SINR in
this notebook:

- TX power: `tx_power_dbm = 51.13` dBm.
- Noise figure: `noise_figure_db = 5.0` dB.
- OFDM configuration from `cfr_mamimo.ipynb`:
  - CP length = 288 samples
  - Bandwidth = 100 MHz
  - Subcarrier spacing = 30 kHz
  - FFT size / subcarriers = 4096
  - Time steps per slot = 14
  - Number of resource blocks = 273

To compute absolute SNR/SINR in Phase G1.1, the following must be resolved:

1. Active subcarrier count / resource-block allocation.
2. Per-subcarrier / per-RB power allocation from 51.13 dBm total TX power.
3. Thermal noise per 30 kHz subcarrier: `k * T * 30 kHz` with `T = 290 K` and
   `NF = 5 dB`.
4. Relationship between the single center-frequency CFR used here and the full
   OFDM bandwidth used in the reference notebook.
5. Whether the reference's `sigma2_dBm = -174 + 10*log10(30e3) + 5` formula
   should be applied per-subcarrier or aggregated.